In [1]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(0.5)

producer.flush()
producer.close()

Writing producer.py


In [3]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    if random.random() < 0.05:
      
        return {
            'tx_id': f'TX{random.randint(1000,9999)}',
            'user_id': f'u{random.randint(1,20):02d}',
            'amount': round(random.uniform(3000.01, 5000.0), 2),  
            'store': random.choice(sklepy),
            'category': 'elektronika',
            'hour': random.randint(0, 5),  
            'timestamp': datetime.now().isoformat(),
        }
    else:
        
        return {
            'tx_id': f'TX{random.randint(1000,9999)}',
            'user_id': f'u{random.randint(1,20):02d}',
            'amount': round(random.uniform(5.0, 3000.0), 2),
            'store': random.choice(sklepy),
            'category': random.choice(kategorie),
            'hour': random.randint(6, 23),  
            'timestamp': datetime.now().isoformat(),
        }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']} | godz: {tx['hour']}")
    time.sleep(0.5)

producer.flush()
producer.close()

Overwriting producer.py


In [4]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
    tx = message.value
    
    if tx['amount'] > 1000:
        print(
            f" ALERT | {tx['tx_id']} | "
            f"{tx['amount']:.2f} PLN | "
            f"{tx['store']} | "
            f"kategoria: {tx['category']} | "
            f"godz: {tx.get('hour', 'brak')}"
        )

Writing consumer_filter.py


In [5]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

def get_risk_level(amount):
    if amount > 3000:
        return "HIGH"
    elif amount > 1000:
        return "MEDIUM"
    else:
        return "LOW"

for message in consumer:
    tx = message.value
    

    tx['risk_level'] = get_risk_level(tx['amount'])
    
    print(
        f"{tx['tx_id']} | {tx['amount']:.2f} PLN | "
        f"{tx['store']} | risk: {tx['risk_level']}"
    )

Writing consumer_enrich.py


In [8]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter, defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='count-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = defaultdict(float)
msg_count = 0

for message in consumer:
    tx = message.value
    store = tx['store']
    amount = tx['amount']
    

    store_counts[store] += 1
    
   
    total_amount[store] += amount
    
    msg_count += 1

    
    if msg_count % 10 == 0:
        print("\n PODSUMOWANIE (co 10 wiadomości):")
        print("-" * 50)
        print(f"{'Sklep':<12} | {'Liczba':<6} | {'Suma (PLN)':<12}")
        print("-" * 50)
        
        for store in store_counts:
            print(f"{store:<12} | {store_counts[store]:<6} | {total_amount[store]:<12.2f}")
        
        print("-" * 50)

Overwriting consumer_count.py


In [7]:
def score_transaction(tx):
    score = 0
    rules = []
    
    amount = tx.get('amount', 0)
    category = tx.get('category', '')
    
 
    hour = int(tx.get('timestamp', '1970-01-01T00:00:00')[11:13])
    
    
    if amount > 3000:
        score += 3
        rules.append('R1')
    
   
    if category == 'elektronika' and amount > 1500:
        score += 2
        rules.append('R2')
    
    
    if hour < 6:
        score += 2
        rules.append('R3')
    
    return score, rules



test_tx = {
    'tx_id': 'TX999',
    'amount': 4500.0,
    'category': 'elektronika',
    'timestamp': '2026-04-01T03:15:00'
}

print(score_transaction(test_tx))

(7, ['R1', 'R2', 'R3'])


In [9]:
%%file scoring_consumer.py
from kafka import KafkaConsumer, KafkaProducer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='scoring-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

alert_producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

def score_transaction(tx):
    score = 0
    rules = []
    
    amount = tx.get('amount', 0)
    category = tx.get('category', '')
    hour = int(tx.get('timestamp', '1970-01-01T00:00:00')[11:13])
    
    if amount > 3000:
        score += 3
        rules.append('R1')
        
    if category == 'elektronika' and amount > 1500:
        score += 2
        rules.append('R2')
        
    if hour < 6:
        score += 2
        rules.append('R3')
    
    return score, rules


for message in consumer:
    tx = message.value
    
    score, rules = score_transaction(tx)
    
    if score >= 3:
        
        tx['score'] = score
        tx['rules_triggered'] = rules
        tx['alert'] = True
        
        
        alert_producer.send('alerts', value=tx)
        
        print(
            f" ALERT | {tx['tx_id']} | "
            f"{tx['amount']:.2f} PLN | "
            f"score: {score} | rules: {rules}"
        )

alert_producer.flush()
alert_producer.close()

Writing scoring_consumer.py
